In [5]:
# %pip install pdfplumber pdfminer.six

import pdfplumber
import re
import os

In [6]:
def extract_full_text(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text.append(page_text)
    return "\n".join(text)


def extract_mda_section(full_text):
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', full_text)

    # Define start and end patterns (case-insensitive)
    start_pattern = re.compile(
        r'item\s*2\.?\s*management[’\'s]{0,2}\s*discussion\s*and\s*analysis', #Item 2. Management's Discussion and Analysis of Financial Condition and Results of Operations
        re.IGNORECASE
    )
    
    end_pattern = re.compile(
        r'item\s*3\.?\s*quantitative\s*and\s*qualitative\s*disclosures', #Quantitative and Qualitative Disclosures About Market Risk 
        re.IGNORECASE
    )

    start_match = start_pattern.search(text)
    if not start_match:
        raise ValueError("MD&A start not found")
    
    matches = list(start_pattern.finditer(text))
    start_index = matches[1].start() ## skip the content page
    end_match = end_pattern.search(text[start_index:])
    if not end_match:
        raise ValueError("Item 3 not found (MD&A end boundary)")

    end_index = start_index + end_match.start()

    mda_text = text[start_index:end_index]
    return mda_text

def save_mda_to_txt(mda_text, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(mda_text)

def extract_mda_from_10q(pdf_path, output_txt_path):
    full_text = extract_full_text(pdf_path)
    mda_text = extract_mda_section(full_text)
    save_mda_to_txt(mda_text, output_txt_path)


In [7]:
def extract_mda_from_all_10q_in_folder(folder_path, output_folder):
    #NVIDIA Corporation_10-Q_2025-10-26T00_00_00_English.pdf
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            output_txt_path = os.path.join(output_folder, filename.replace(".pdf", "_MDA.txt"))
            try:
                extract_mda_from_10q(pdf_path, output_txt_path)
            except Exception as e:
                raise e

In [8]:
## run here after all 10-q files are in the "10-Q" folder
folder_path = "10-Q"
output_folder = "Data"
extract_mda_from_all_10q_in_folder(folder_path, output_folder)